# ceol-gpt — Memorization Check

Checks whether generated tunes are novel or reproduced verbatim from training data.

**Longest contiguous match fraction** is the key metric:
- `< 30%` → sharing common phrases (normal, model is generalizing)
- `30–80%` → heavy copying, worth investigating
- `≥ 80%` → likely memorized verbatim

**Before running:** make sure you have a trained checkpoint in `models/{RUN_NAME}/best.pt`.

## 0 — Mount Drive & clone repo

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

# ── Edit this path to wherever you keep the repo in your Drive ───────────────
REPO_PATH = '/content/drive/MyDrive/Duke/cs372/Project/ceol-gpt'
GITHUB_URL = 'https://github.com/andrewmcknight/ceol-gpt.git'
# ─────────────────────────────────────────────────────────────────────────────

if os.path.isdir(REPO_PATH):
    print('Repo already exists — pulling latest changes')
    !git -C {REPO_PATH} pull
else:
    print('Cloning repo into Drive')
    !git clone {GITHUB_URL} {REPO_PATH}

%cd {REPO_PATH}
import sys
if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

!ls

Repo already exists — pulling latest changes
remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 2.14 KiB | 2.00 KiB/s, done.
From https://github.com/andrewmcknight/ceol-gpt
   423d140..fe50b6e  main       -> origin/main
Updating 423d140..fe50b6e
Fast-forward
 src/evaluate.py | 205 ++++++++++++++++++++++++--------------------------------
 1 file changed, 89 insertions(+), 116 deletions(-)
/content/drive/MyDrive/Duke/cs372/Project/ceol-gpt
ATTRIBUTION.md	configs  docs	 notebooks  requirements.txt  src
CLAUDE.md	data	 models  README.md  SETUP.md	      videos


## 1 — Install dependencies

In [3]:
%pip install -q -r requirements.txt

## 2 — Config

In [4]:
CONFIG   = 'configs/small.yaml'   # must match the config used to train
RUN_NAME = 'small'                # checkpoint loaded from models/{RUN_NAME}/best.pt

TEMPERATURE    = 0.8
TOP_K          = 50
MAX_NEW_TOKENS = 256
N_CHECK        = 20   # number of tunes to generate and check

print(f'Config:      {CONFIG}')
print(f'Checkpoint:  models/{RUN_NAME}/best.pt')

Config:      configs/small.yaml
Checkpoint:  models/small/best.pt


## 3 — Load model & tokenizer

In [5]:
import torch
import yaml
from src.model import build_model
from src.tokenizer import ABCTokenizer

device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ckpt_path = f'models/{RUN_NAME}/best.pt'

with open(CONFIG) as f:
    cfg = yaml.safe_load(f)

tokenizer = ABCTokenizer.load(f'models/{RUN_NAME}/tokenizer.pkl')
model     = build_model(cfg, vocab_size=len(tokenizer)).to(device)
ckpt      = torch.load(ckpt_path, map_location=device)
model.load_state_dict(ckpt['model_state'])
model.eval()
print(f'Device:     {device}')
print(f'Checkpoint: epoch {ckpt["epoch"] + 1}, val loss {ckpt["val_loss"]:.4f}')

Device:     cuda
Checkpoint: epoch 95, val loss 1.0897


## 4 — Generate & check for memorization

In [6]:
import json
import random
import torch.nn.functional as F
from src.evaluate import memorization_report


def generate(model, tokenizer, tune_type, key, meter,
             temperature=0.8, top_k=50, max_new_tokens=256):
    prompt_ids = tokenizer.encode('', tune_type, key, meter, add_eos=False)
    ids = torch.tensor(prompt_ids, dtype=torch.long, device=device).unsqueeze(0)
    with torch.no_grad():
        for _ in range(max_new_tokens):
            logits = model(ids)[:, -1, :]
            if top_k > 0:
                vals, _ = torch.topk(logits, top_k)
                logits[logits < vals[:, -1:]] = float('-inf')
            probs   = F.softmax(logits / temperature, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            if next_id.item() == tokenizer.eos_id:
                break
            ids = torch.cat([ids, next_id], dim=1)
    return tokenizer.decode_to_abc(ids[0].tolist())


with open('data/tunes.json') as f:
    all_tunes = json.load(f)

check_configs = (
    [('reel',     'Gmajor', '4/4')] * 7 +
    [('jig',      'Gmajor', '6/8')] * 7 +
    [('hornpipe', 'Dmajor', '4/4')] * 6
)
random.shuffle(check_configs)

generated_abc = []
for tune_type, key, meter in check_configs[:N_CHECK]:
    abc = generate(model, tokenizer, tune_type, key, meter,
                   temperature=TEMPERATURE, top_k=TOP_K,
                   max_new_tokens=MAX_NEW_TOKENS)
    generated_abc.append(abc)

results = memorization_report(generated_abc, all_tunes, verbose=True)

Tokenizing 54,246 training tunes (k=10)...
  Unique 10-gram hashes: 4,962,636

  Memorization Report
  Generated tunes analyzed : 20
  K-gram size              : 10
  Exact matches            : 0 (0.0%)
  Avg 10-gram coverage      : 22.3%
  Max 10-gram coverage      : 45.3%

    #  tokens  coverage  exact
  ---  ------  --------  -----
    0     104    45.3%     no
    1     173    19.5%     no
    2     137    21.9%     no
    3     108     6.1%     no
    4     249    13.8%     no
    5     102    21.5%     no
    6     117     6.5%     no
    7     146     6.6%     no
    8      95    36.0%     no
    9     138    27.9%     no
   10     148    28.8%     no
   11     147    33.3%     no
   12     152    16.8%     no
   13     254    35.5%     no
   14     256    10.1%     no
   15     123    14.9%     no
   16     256    20.6%     no
   17     100     3.3%     no
   18     123    39.5%     no
   19     129    38.3%     no

  Interpretation (k=10):
    coverage >= 70%  → heavily deriv